# AutoGen Studio and Low-Code Prototyping

**AutoGen Studio** is Microsoft's browser-based UI for building multi-agent workflows on top of the **AgentChat API**. Drag agents, wire teams, attach tools, and run tasks — then **export to Python** when the prototype is ready.

This notebook covers installation, launching the workflow UI, Studio vs hand-written code, export workflows, limitations, dev vs production boundaries, UI walkthroughs (described in text), and links to official Microsoft documentation.

Prerequisites: **02. Setup and the AgentChat API**, **08. GroupChat and Multi-Agent Teams**.


In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import asyncio
import json
import os
import re
from dataclasses import dataclass, field
from typing import Any, Sequence

try:
    import autogen_agentchat
    print("autogen-agentchat:", getattr(autogen_agentchat, "__version__", "installed"))
except ImportError:
    print("autogen-agentchat: pip install autogen-agentchat autogen-ext[openai]")

print("asyncio loop ready — use await in notebook cells")


---

## 1. What AutoGen Studio Is

AutoGen Studio is a **low-code IDE** for AgentChat teams. It targets **prototyping** — not production deployment.

```
┌─────────────────────────────────────────────────────────────┐
│  AutoGen Studio (browser @ localhost:8081)                  │
│  ┌─────────┐  ┌─────────┐  ┌──────────┐  ┌─────────────┐  │
│  │ Agents  │  │  Teams  │  │  Tools   │  │    Playground│  │
│  │  panel  │  │ builder │  │ registry │  │    (run task)│  │
│  └────┬────┘  └────┬────┘  └────┬─────┘  └──────┬──────┘  │
│       └────────────┴────────────┴────────────────┘         │
│                         │                                    │
│                         ▼                                    │
│              AgentChat runtime (same as code)                │
└─────────────────────────────────────────────────────────────┘
```

Official docs: [AutoGen Studio User Guide](https://microsoft.github.io/autogen/stable/user-guide/autogenstudio-user-guide/index.html)


---

## 2. Installation

Requires **Python 3.10+**. Use a virtual environment:


In [ ]:
# Recommended install (run in terminal, not required in notebook)
INSTALL_STEPS = """
python -m venv .venv
# Windows:
.venv\\Scripts\\activate
# macOS/Linux:
source .venv/bin/activate

pip install -U autogenstudio
pip install -U "autogen-agentchat" "autogen-ext[openai]"
"""
print(INSTALL_STEPS)


PyPI package: `autogenstudio`. It bundles the web UI and persists workflows under `--appdir`.

Docs: [Installation](https://microsoft.github.io/autogen/stable/user-guide/autogenstudio-user-guide/installation.html)


---

## 3. Launching the UI


In [ ]:
LAUNCH = """
# Default port 8081
autogenstudio ui --port 8081

# Custom data directory (teams, sessions, settings)
autogenstudio ui --port 8081 --appdir ./autogen-studio-data
"""
print(LAUNCH)
print("Open: http://localhost:8081/")


**Described screenshot — Home screen:** After launch, the browser shows a dark sidebar with **Build**, **Playground**, and **Gallery** tabs. The center panel welcomes you to AutoGen Studio v0.4 with a **New Team** button and cards for sample workflows.


---

## 4. Workflow UI Walkthrough

### 4a. Agents Panel

**Described screenshot:** Left column lists agent cards. Each card shows **Name**, **Model** dropdown (e.g. `gpt-4o-mini`), and a **System Message** text area. A **+ Add Agent** button sits at the top.

For the Notes API curriculum, create:
- `docs_writer` — guard prompt from **13**
- `critic` — reviews and says `APPROVE`


### 4b. Teams Panel

**Described screenshot:** Center canvas shows a **RoundRobin** or **Selector** team node with agent avatars connected in a ring. A **Termination** section offers dropdowns for `MaxMessageTermination` and `TextMentionTermination` with text field `APPROVE`.


### 4c. Tools Panel

**Described screenshot:** Right drawer lists registered Python functions or OpenAPI tools. For Notes API, add a `search_docs` function that queries `NOTES_CORPUS` — same stub as **06. Tools and Function Registration**.


### 4d. Playground

**Described screenshot:** Chat-style interface. User types *"How do I run Alembic migrations?"* at the bottom. Messages stream in with `docs_writer` and `critic` labels. A **Stop** button maps to `ExternalTermination` (**13**).


---

## 5. When to Use Studio vs Code

| Situation | Use Studio | Use Code |
|-----------|------------|----------|
| First team layout | ✓ drag-and-drop | |
| Prompt iteration with PM | ✓ live playground | |
| Custom termination logic | | ✓ `|` / `&` combos |
| CI/CD deployment | | ✓ `build_notes_team()` |
| Auth + rate limits | | ✓ FastAPI layer (**16**) |
| Version control | export → commit | ✓ native git |


In [ ]:
DECISION = {
    "prototype_new_roles": "Studio",
    "production_service": "Code (16)",
    "complex_selector_prompt": "Code (09)",
    "demo_to_stakeholder": "Studio Playground",
}
print(json.dumps(DECISION, indent=2))


---

## 6. Export to Python

Studio can export a team definition as runnable AgentChat code. Typical export structure:


In [ ]:
EXPORTED_SKETCH = '''
# Exported from AutoGen Studio — review before production use
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

docs_writer = AssistantAgent(
    "docs_writer",
    model_client=model_client,
    system_message="...",  # from Studio UI
)
critic = AssistantAgent(
    "critic",
    model_client=model_client,
    system_message="...",
)

team = RoundRobinGroupChat(
    [docs_writer, critic],
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(12),
)
'''
print(EXPORTED_SKETCH)


**Post-export checklist:**

1. Replace hardcoded API keys with `os.environ` / `pydantic-settings`
2. Add `is_blocked()` pre-flight guard (**13**)
3. Add `TokenUsageTermination` if not in export
4. Move factory into `build_notes_team()` (**16**)
5. Add tests for termination and blocked prompts


---

## 7. Limitations

| Limitation | Impact | Mitigation |
|------------|--------|------------|
| **Not production-ready** | No auth, no multi-tenant | Deploy via FastAPI (**16**) |
| **Local filesystem state** | `--appdir` not shared across pods | Export code + env config |
| **Limited CI** | UI workflows hard to diff | Export → git |
| **Selector tuning** | Model-based speaker selection opaque | Debug in code (**09**, **15**) |
| **Security** | Tools run with Studio process privileges | Sandboxing (**07**) |

Microsoft explicitly states Studio is for **rapid prototyping** — see [Security Note](https://microsoft.github.io/autogen/stable/user-guide/autogenstudio-user-guide/index.html#a-note-on-security).


---

## 8. Development vs Production

```
DEV (Studio)                    PROD (Code)
─────────────                   ────────────
autogenstudio ui                uvicorn app:app
localhost:8081                  HTTPS + auth
manual Playground               POST /chat API
ad-hoc API keys                 secrets manager
no termination audit            logged TaskResult
single user                     rate limits + quotas
```


In [ ]:
ENV_MATRIX = [
    ("OPENAI_API_KEY", "dev: .env file", "prod: vault / K8s secret"),
    ("APP_MAX_TEAM_TOKENS", "optional", "required (13)"),
    ("APP_MODEL", "gpt-4o-mini", "tiered by user plan"),
    ("AUTOGENSTUDIO", "ui --port 8081", "not deployed"),
]
for var, dev, prod in ENV_MATRIX:
    print(f"{var:25} dev={dev:25} prod={prod}")


---

## 9. Notes API Studio Recipe

Step-by-step to replicate our curriculum team in Studio:

1. **New Team** → RoundRobin
2. Add `docs_writer` with Notes API guard prompt
3. Add `critic` with `APPROVE` instruction
4. Termination: `TextMentionTermination(APPROVE)` + `MaxMessageTermination(12)`
5. Playground task: *"Explain JWT authentication for the Notes API"*
6. Export → save as `notes_team_exported.py`
7. Refactor into `build_notes_team()` in **16**


In [ ]:
# Notes API documentation corpus (shared across 03. AutoGen/ track)
NOTES_CORPUS = [
    {"id": "c1", "text": "The Notes API is built with FastAPI. Routes live under /notes with GET, POST, PUT, DELETE."},
    {"id": "c2", "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling revisions."},
    {"id": "c3", "text": "JWT bearer tokens authenticate API requests. Send Authorization: Bearer token header."},
    {"id": "c4", "text": "Pytest fixtures share database setup. Use conftest.py for session-scoped engines."},
    {"id": "c5", "text": "Alembic autogenerate compares SQLAlchemy models to the live schema and drafts revision files."},
]

BLOCKED_TERMS = {"password", "secret", "api key", "private key", "credential"}
APPROVE_TOKEN = "APPROVE"
TERMINATE_TOKEN = "TERMINATE"


def keyword_search(query: str, k: int = 2) -> str:
    """Simple keyword retrieval over NOTES_CORPUS."""
    tokens = set(query.lower().split())
    scored = [(len(tokens & set(d["text"].lower().split())), d) for d in NOTES_CORPUS]
    scored.sort(key=lambda x: x[0], reverse=True)
    top = [d for s, d in scored if s > 0][:k] or [NOTES_CORPUS[0]]
    return "\n".join(f"[{d['id']}] {d['text']}" for d in top)


def is_blocked(text: str) -> bool:
    """Pre-flight guard — mirror 02. LangGraph/14 guard node."""
    lower = text.lower()
    return any(term in lower for term in BLOCKED_TERMS)


print("corpus chunks:", len(NOTES_CORPUS))
print("sample search:", keyword_search("JWT bearer")[:80], "...")


Register `keyword_search` as a tool in Studio's Tools panel so the writer can retrieve corpus chunks during Playground runs.


---

## 10. Gallery and Templates

**Described screenshot — Gallery tab:** Pre-built templates include **RoundRobin Critic**, **Selector Research**, and **Sequential Pipeline**. Click **Use Template** to clone into your workspace.

Templates accelerate Notes API prototyping — start from **RoundRobin Critic** and swap system messages.


---

## 11. Integrating Studio with the Code Track

| Studio artifact | Code destination |
|-----------------|------------------|
| Agent system messages | `GUARD_PROMPT` constants |
| Team topology | `RoundRobinGroupChat` / `SelectorGroupChat` |
| Termination settings | `production_termination` (**13**) |
| Tool definitions | `@tool` functions (**06**) |
| Playground transcripts | Test fixtures for **15** |


In [ ]:
# Map Studio export fields to production factory (16)
STUDIO_TO_PROD = {
    "agents[].system_message": "GUARD_PROMPT / critic prompt",
    "team.type": "RoundRobinGroupChat vs SelectorGroupChat",
    "termination": "TextMentionTermination | MaxMessageTermination",
    "tools[]": "registered FunctionTool wrappers",
}
print(json.dumps(STUDIO_TO_PROD, indent=2))


---

## 12. Troubleshooting

| Issue | Fix |
|-------|-----|
| Port in use | `autogenstudio ui --port 8082` |
| Blank UI after upgrade | Clear `--appdir` cache or reinstall |
| Model errors | Set `OPENAI_API_KEY` before launch |
| Export missing termination | Add manually in code (**13**) |
| Playground infinite loop | Add `MaxMessageTermination` in Team settings |


---

## 13. Official Documentation Links

| Resource | URL |
|----------|-----|
| **Studio User Guide** | https://microsoft.github.io/autogen/stable/user-guide/autogenstudio-user-guide/index.html |
| **Installation** | https://microsoft.github.io/autogen/stable/user-guide/autogenstudio-user-guide/installation.html |
| **AgentChat Tutorial** | https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/index.html |
| **Termination** | https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/tutorial/termination.html |
| **GitHub — autogen-studio** | https://github.com/microsoft/autogen/tree/main/python/packages/autogen-studio |
| **Video walkthrough (v0.4)** | https://youtu.be/oum6EI7wohM |


---

## 14. Summary

| Takeaway | Detail |
|----------|--------|
| **Studio = prototype** | Fast agent/team iteration in the browser |
| **Export early** | Move to code before adding auth or scaling |
| **Same runtime** | Studio uses AgentChat — patterns from **02–13** apply |
| **Not for prod** | Use **16** for FastAPI service layer |
| **Next** | **15** — observe and debug runs; **16** — production capstone |

Next: **15. Observability and Debugging Multi-Agent Runs**.
